## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from src.cryptography.cnn_bfv import BFVFHE
from src.utils.notebook_helper import encrypt_test_data, test_ipfe_cnn, test_regular_ipfe_cnn, load_data


In [2]:
base_model = 1
model_path = f"models/cnn_model_{base_model}.pth"

## Define Model

In [21]:
class FHECNN(nn.Module):
    def __init__(self, device, num_classes=10):
        super(FHECNN, self).__init__()
        self.device = device

        # First convolutional block (same as IPFECNN)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Second convolutional block
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Third convolutional block
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 1 * 1, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

        # Load trained weights (same checkpoint as IPFECNN)
        self.load_state_dict(torch.load(model_path, map_location=device))
        print("weights copied from trained model")

        # Cache conv1 weights and biases for FHE inner products
        w = self.conv1.weight.data  # (16, 1, 3, 3)
        self.kernels = torch.round(w.view(w.size(0), -1).squeeze(1).view(w.size(0), -1) * 10000).long().tolist()
        self.biases = self.conv1.bias.detach().cpu()         # (16,)

        # TenSEAL CKKS context (toy params; tune later)
        self.fhe = BFVFHE()
        self.fhe.setup(9, n_length=28)

    # ---------- FHE analog of encrypt_data ----------
    def encrypt_data(self, test_set):
        unfold = nn.Unfold(kernel_size=3, stride=3, padding=1)
        patches = unfold(test_set)  # (B, 9, num_patches)
        B, patch_size, num_patches = patches.shape

        encrypted_patches = []
        for b in range(B):
            patches_b = patches[b].T  # (num_patches, 9)
            encrypted_image = []
            for p in range(num_patches):
                patch = patches_b[p]  # (9,)
                ct_patch = self.fhe.encrypt(patch)
                encrypted_image.append(ct_patch)
            encrypted_patches.append(encrypted_image)

        return encrypted_patches


    # ---------- FHE analog of first_conv_forward ----------

    def first_conv_forward(self, x, H, W):
        # x: list of encrypted patches for one image
        num_patches = len(x)
        num_kernels = len(self.kernels)
        decrypted_maps = torch.zeros(num_kernels, num_patches, device=self.device)

        for k in range(num_kernels):
            k_vec = self.kernels[k]          # torch length-9
            bias_k = float(self.biases[k].item())
            for p in range(num_patches):
                ct_patch = x[p]
                ip = self.fhe.inner_product(ct_patch, k_vec) / 10000
                decrypted_maps[k, p] = ip + bias_k

        return decrypted_maps.view(1, num_kernels, H, W)


    # ---------- Forward (same interface as IPFECNN) ----------

    def forward(self, x, H, W, encrypted=False):
        """
        If encrypted=False:
            x: torch.Tensor of shape (B, 1, H_in, W_in)
        If encrypted=True:
            x: list of length B; each element is a list of encrypted patches for that image.
        H, W: conv1 output spatial dims (must match encrypt_data layout).
        """
        if encrypted:
            outputs = []
            for sample in x:   # sample is [ct_patch_0, ..., ct_patch_{num_patches-1}]
                feat = self.first_conv_forward(sample, H, W)   # (1, 16, H, W)
                feat = self.pool1(F.relu(self.bn1(feat.to(self.device))))
                feat = self.pool2(F.relu(self.bn2(self.conv2(feat))))
                feat = self.pool3(F.relu(self.bn3(self.conv3(feat))))
                feat = feat.view(feat.size(0), -1)
                feat = F.relu(self.fc1(feat))
                feat = self.dropout(feat)
                feat = self.fc2(feat)
                outputs.append(feat)
            return torch.cat(outputs, dim=0)
        else:
            x = self.conv1(x)
            x = self.pool1(F.relu(self.bn1(x)))
            x = self.pool2(F.relu(self.bn2(self.conv2(x))))
            x = self.pool3(F.relu(self.bn3(self.conv3(x))))
            x = x.view(x.size(0), -1)
            x = F.relu(self.fc1(x))
            x = self.dropout(x)
            x = self.fc2(x)
            return x

## Initialize Model

In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
fhe_model = FHECNN(device=device, num_classes=10).to(device)
print(f"FHE-CNN model created on device: {device}")

weights copied from trained model

=== Paillier Key Info ===
p = 12157
q = 13127
n = 159584939
FHE-CNN model created on device: cpu


In [26]:
test_loader = load_data()
encrypted_data, labels = encrypt_test_data(fhe_model, test_loader, device, num_samples=5)

Test samples: 10000
Encrypted 5 samples.


## Test model

In [27]:
print("Testing CNN functionality...")
test_ipfe_cnn(fhe_model, encrypted_data, labels, H=10, W=10, device=device)


Testing CNN functionality...
Testing CNN forward pass on encrypted data...
Labels of test samples: [7 2 1 0 4]
Predictions on encrypted data: [7 2 1 0 4]
Accuracy on encrypted samples: 100.00% (5/5)


In [16]:
test_regular_ipfe_cnn(fhe_model, test_loader, device, num_samples=5)

Testing CNN forward pass on encrypted data...
Predictions on encrypted data: [7 2 1 0 4]
Accuracy on encrypted samples: 100.00% (5/5)
